# 01.5 Building Models with `nn.Module`

This notebook is where we start working with actual models.

`nn.Module` is the core abstraction behind `PyTorch` models.

Key concepts:

- module
- layer
- forward pass
- parameters
- submodules
- multilayer perceptron, MLP

## Learning Goals

After this notebook, you should be able to:

1. Understand the basic structure of `nn.Module`.
2. Define simple model classes yourself.
3. Distinguish layers from parameters.
4. Read what `forward` is doing.
5. Build a simple MLP.
6. Prepare the model component for later training loops.

In [ ]:
import torch
import torch.nn as nn

## The Minimal Structure of `nn.Module`

A minimal model usually has two parts:

1. define layers
2. define how data flows through the layers

In [ ]:
class SimpleLinearModel(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.linear(x)


model = SimpleLinearModel(in_features=2, out_features=1)
print(model)

The meaning here is very direct:

- input 2 features, output 1 value
- when data comes in, pass it through this layer

## Forward Pass

`forward` defines how inputs become outputs.

In `PyTorch`, you usually do not write `model.forward(x)` directly. You write `model(x)`.


In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
out = model(x)

print("x.shape =", x.shape)
print("out =\n", out)
print("out.shape =", out.shape)

Because `out_features=1`, the output shape is `(batch_size, 1)`.


In [ ]:
# Exercise 1
# Create a linear model with input dimension 3 and output dimension 2.
#Then feed in a tensor with shape (4, 3)  tensor and print the output shape.
# Then feed in a tensor of shape (4, 3) and print the output shape.

# model_ex =
# x_ex =
# out_ex =
# print(out_ex.shape)

In [ ]:
# Exercise 1 Reference Solution

model_ex = SimpleLinearModel(in_features=3, out_features=2)
x_ex = torch.randn(4, 3)
out_ex = model_ex(x_ex)
print(out_ex.shape)

## Parameters

A model can learn because it contains updateable parameters.

For `nn.Linear`, the most typical parameters are:

- weights
- bias

In [ ]:
for name, param in model.named_parameters():
    print(name)
    print("shape =", param.shape)
    print(param)
    print()

When you later see `model.parameters()` or `model.named_parameters()`, you are essentially looking at the learnable parts of the model.


## Adding an Activation Function

With only linear layers, the model has limited expressive power.

That is why non-linear activation functions are often inserted between linear layers.

Common examples:

- `ReLU`
- `Sigmoid`
- `Tanh`

In [ ]:
class TwoLayerMLP(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_features, out_features)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


mlp = TwoLayerMLP(in_features=2, hidden_features=4, out_features=1)
print(mlp)

In [ ]:
x = torch.randn(5, 2)
out = mlp(x)
print("x.shape =", x.shape)
print("out.shape =", out.shape)
print(out)

Shape flow here:

- input: `(5, 2)`
- after first layer: `(5, 4)`
- after second layer: `(5, 1)`

Understanding shape flow is extremely important.


In [ ]:
# Exercise 2
# Define a model:
# input dim = 4
# hidden dim = 8
# 
# Then feed in a tensor with shape (6, 4)  tensor and print the output shape.

# model2 =
# x2 =
# out2 =
# print(out2.shape)

In [ ]:
# Exercise 2 Reference Solution

model2 = TwoLayerMLP(in_features=4, hidden_features=8, out_features=3)
x2 = torch.randn(6, 4)
out2 = model2(x2)
print(out2.shape)

## 5. `nn.Sequential`

If the model is a simple chain of layers, `nn.Sequential` can make the code more compact.


In [ ]:
seq_model = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
)

print(seq_model)

x = torch.randn(3, 2)
out = seq_model(x)
print("out.shape =", out.shape)

`nn.Sequential` is great for simple chains, but when you need branches, skip connections, or multiple inputs/outputs, a custom `forward` is usually clearer.


## Output Layer and Task Type

The design of the final layer depends strongly on the task type.

Common cases:

- output one or more continuous values
- output a logit or probability
- output one score per class

This becomes even more important in the next notebook on loss functions.


In [ ]:
reg_model = TwoLayerMLP(in_features=2, hidden_features=4, out_features=1)
cls_model = TwoLayerMLP(in_features=2, hidden_features=4, out_features=3)

x = torch.randn(4, 2)
print("regression output shape =", reg_model(x).shape)
print("classification output shape =", cls_model(x).shape)

In [ ]:
# Exercise 3
# For each task below, decide what the output dimension is usually.
#1. predict house price / predict house price
# spam or not spam
# 
# Write your answer in one sentence.


Reference answer:

- usually output dimension 1
- often output dimension 1 or 2, depending on the loss setup
- usually output dimension 10

## Summary

The key lesson here is not memorizing class names, but understanding the fixed structure of model definition:

1. define layers in `__init__`
2. define data flow in `forward`
3. let the model learn through parameters

You should now be able to answer:

1. Why is `nn.Module` the core abstraction for models?
2. Why are layers usually defined in `__init__` instead of inside `forward`?
3. Why does output dimension depend on task type?
4. When is `nn.Sequential` suitable and when is a custom `forward` better?

Suggested next step:

- Move to the loss-and-optimizer notebook to connect model outputs to learning.